In [1]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# ==========================================
# 0. GLOBAL PLOTTING & ENVIRONMENT SETUP
# ==========================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

# Get current working directory (e.g., .\notebooks)
cwd = os.getcwd()

# Get parent folder to access (.\Data)
parent = os.path.dirname(cwd)

# Get file path to load into environment
file_path_combined_data = os.path.join(parent, "Data", "combined_data.parquet")

print(f"Loading dataset from: {file_path_combined_data}")
df = pl.read_parquet(file_path_combined_data)

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\combined_data.parquet


In [2]:
# ==========================================
# 1. FEATURE CLEANING & COLLINEARITY RESOLUTION
# ==========================================
print("Cleaning features and handling multicollinearity...")

# Drop 'Price_signal' to eliminate string concatenation redundancy & collinearity 
# with 'Price_profile' and 'Price_level', avoiding complex string parsing overhead.
if "Price_signal" in df.columns:
    df = df.drop("Price_signal")
    print("-> Dropped 'Price_signal' column.")

# Filter dataset to Phase 2 active rows for robust modeling and analysis
df_phase2 = df.filter(
    (pl.col("Participation_Phase") == "Phase_2") & 
    (pl.col("Participation_status_Phase2") == "OK")
)

print(f"Loaded Phase 2 active rows: {df_phase2.height:,}")

Cleaning features and handling multicollinearity...
-> Dropped 'Price_signal' column.
Loaded Phase 2 active rows: 11,996,256


In [5]:
# ==========================================
# 2. FEATURE ENGINEERING: TEMPORAL & WEATHER
# ==========================================
# Add calendar features (Day of Week, Weekend indicator, Sunny conditions proxy if available)
df_phase2 = df_phase2.with_columns(
    Parsed_Date = pl.col("Date"), #.str.to_date("%Y-%m-%d"),
).with_columns(
    Day_Of_Week = pl.col("Parsed_Date").dt.weekday(), # 1 = Monday, 7 = Sunday
).with_columns(
    Is_Weekend = pl.when(pl.col("Day_Of_Week").is_in([6, 7])).then(1).otherwise(0).cast(pl.Int32),
    Is_Sunny = pl.when(pl.col("Temperature24") > pl.col("Temperature24").mean()).then(1).otherwise(0).cast(pl.Int32)
)

In [6]:
# ==========================================
# 3. TARGET ENGINEERING: EXTENDED FPI MODEL
# ==========================================
print("Engineering Flexibility Propensity Index (FPI) with weather/weekend interactions...")

# Step A: Estimate household baseline consumption on non-experiment weekdays by hour & temperature bin
df_baselines = (
    df_phase2
    .filter(pl.col("Experiment_price_NOK_kWh").is_null())
    .with_columns(
        Temp_Bin = (pl.col("Temperature24") / 5).round(0) * 5 # Bin temperature by 5°C increments
    )
    .group_by(["ID", "Hour", "Temp_Bin"])
    .agg(
        Baseline_kWh = pl.col("Demand_kWh").mean()
    )
)

# Step B: Join baseline back to experiment days
df_experiment_days = (
    df_phase2
    .filter(pl.col("Experiment_price_NOK_kWh").is_not_null())
    .with_columns(
        Temp_Bin = (pl.col("Temperature24") / 5).round(0) * 5
    )
    .join(df_baselines, on=["ID", "Hour", "Temp_Bin"], how="left")
    .with_columns(
        Baseline_kWh = pl.col("Baseline_kWh").fill_null(pl.col("Demand_kWh").mean())
    )
)

# Step C: Aggregate to Household x Experiment-Day Level including interaction terms
df_fpi_daily = (
    df_experiment_days
    .group_by(["ID", "Date", "Region", "Price_profile", "Price_level", "Is_Weekend"])
    .agg(
        Actual_Peak_Demand = pl.col("Demand_kWh").sum(),
        Base_Peak_Demand = pl.col("Baseline_kWh").sum(),
        Mean_Temp = pl.col("Temperature").mean(),
        Mean_Sunny = pl.col("Is_Sunny").mean()
    )
    .with_columns(
        # Flexibility Propensity Index formula capturing demand shift/reduction
        Flexibility_Propensity_Index = (
            (pl.col("Base_Peak_Demand") - pl.col("Actual_Peak_Demand")) / (pl.col("Base_Peak_Demand") + 1e-6) * 100
        ).clip(-50.0, 50.0),
        
        # Interaction terms for weekend temperature sensitivity
        Temp_Weekend_Interaction = pl.col("Mean_Temp") * pl.col("Is_Weekend")
    )
)

print(f"Computed FPI across {df_fpi_daily.height:,} household-experiment days.")

Engineering Flexibility Propensity Index (FPI) with weather/weekend interactions...
Computed FPI across 59,677 household-experiment days.


In [37]:
# ==========================================
# 4. APPENDIX B: GENERATING TABLES & FIGURES
# ==========================================

# --- TABLE B1: Dataset Dimensionality & Observation Breakdown ---
table_b1 = (
    df
    .group_by(["Participation_Phase", "Participation_Experiment", "Control_Price_Phase2"])
    .agg(
        Total_Rows = pl.len(),
        Unique_Households = pl.col("ID").n_unique()
    )
    .sort(["Participation_Phase", "Participation_Experiment"])
)

print("\n=== Table B1: Dataset Dimensionality & Observation Breakdown ===")
print(table_b1)


=== Table B1: Dataset Dimensionality & Observation Breakdown ===
shape: (2, 5)
┌─────────────────────┬─────────────────────┬─────────────────────┬────────────┬───────────────────┐
│ Participation_Phase ┆ Participation_Exper ┆ Control_Price_Phase ┆ Total_Rows ┆ Unique_Households │
│ ---                 ┆ iment               ┆ 2                   ┆ ---        ┆ ---               │
│ str                 ┆ ---                 ┆ ---                 ┆ u32        ┆ u32               │
│                     ┆ str                 ┆ str                 ┆            ┆                   │
╞═════════════════════╪═════════════════════╪═════════════════════╪════════════╪═══════════════════╡
│ Phase_2             ┆ Yes                 ┆ Control group       ┆ 3541248    ┆ 1272              │
│ Phase_2             ┆ Yes                 ┆ Price group         ┆ 8638752    ┆ 3103              │
└─────────────────────┴─────────────────────┴─────────────────────┴────────────┴───────────────────┘


In [39]:
# ==========================================
# IDENTIFYING EXCLUDED HOUSEHOLDS & REASONS
# ==========================================

# 1. Get all unique households in the raw dataset
all_households = df.select("ID").unique()

# 2. Define the final analysis sample households 
# (Phase 2 active treatment 'OK' + Control group with valid status)
analysis_sample = (
    df
    .filter(
        (pl.col("Participation_Phase") == "Phase_2") & 
        (pl.col("Participation_status_Phase2").is_in(["OK", "Control"]))
    )
    .select("ID")
    .unique()
)

# 3. Find excluded households (present in raw data but not in final analysis sample)
excluded_households = all_households.join(analysis_sample, on="ID", how="anti")

print(f"Total raw unique households: {all_households.height}")
print(f"Final analysis sample households: {analysis_sample.height}")
print(f"Total excluded households: {excluded_households.height}")

# 4. Inspect exclusion reasons / status breakdown for excluded IDs
exclusion_breakdown = (
    df
    .filter(pl.col("ID").is_in(excluded_households.get_column("ID")))
    .group_by("Participation_status_Phase2")
    .agg(
        Excluded_Count = pl.col("ID").n_unique()
    )
    .sort("Excluded_Count", descending=True)
)



exclusion_breakdown_ID = (
    df
    .filter(pl.col("ID").is_in(excluded_households.get_column("ID")))
    .group_by("ID")
    .agg(
        Excluded_Count = pl.col("ID").n_unique()
    )
    .sort("Excluded_Count", descending=True)
)

print("\n=== Breakdown of Excluded Households by Status ===")
print(exclusion_breakdown_ID)

Total raw unique households: 4375
Final analysis sample households: 4309
Total excluded households: 66

=== Breakdown of Excluded Households by Status ===
shape: (66, 2)
┌─────────┬────────────────┐
│ ID      ┆ Excluded_Count │
│ ---     ┆ ---            │
│ str     ┆ u32            │
╞═════════╪════════════════╡
│ Exp_359 ┆ 1              │
│ Exp_386 ┆ 1              │
│ Exp_221 ┆ 1              │
│ Exp_225 ┆ 1              │
│ Exp_361 ┆ 1              │
│ …       ┆ …              │
│ Exp_190 ┆ 1              │
│ Exp_338 ┆ 1              │
│ Exp_428 ┆ 1              │
│ Exp_418 ┆ 1              │
│ Exp_306 ┆ 1              │
└─────────┴────────────────┘


C:\Users\Marty\AppData\Local\Temp\ipykernel_52848\1020618743.py:30: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("ID").is_in(excluded_households.get_column("ID")))
C:\Users\Marty\AppData\Local\Temp\ipykernel_52848\1020618743.py:42: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("ID").is_in(excluded_households.get_column("ID")))


In [41]:
# ==========================================
# EXTRACTING EXCLUDED HOUSEHOLD IDS
# ==========================================
print(f"Cleaned DataFrame shape before removal: {df.shape}")
# 1. Get all unique household IDs present in the raw dataset
all_households = df.select("ID").unique()

# 2. Define the final analysis sample households 
# (Phase 2 active treatment 'OK' and valid Control group members)
analysis_sample = (
    df
    .filter(
        (pl.col("Participation_Phase") == "Phase_2") & 
        (pl.col("Participation_status_Phase2").is_in(["OK", "Control"]))
    )
    .select("ID")
    .unique()
)

# 3. Use an anti-join to find households present in the raw data but missing from the analysis sample
excluded_households = all_households.join(analysis_sample, on="ID", how="anti")

# Convert to a Python list for downstream filtering or inspection
excluded_household_list = excluded_households.get_column("ID").to_list()

print(f"Total raw unique households: {all_households.height}")
print(f"Final analysis sample households: {analysis_sample.height}")
print(f"Total excluded households identified: {len(excluded_household_list)}")

# 4. Optional: Filter your main dataframe to completely remove these excluded households
df_cleaned = df.filter(~pl.col("ID").is_in(excluded_household_list))
print(f"Cleaned DataFrame shape after removal: {df_cleaned.shape}")

Cleaned DataFrame shape before removal: (12180000, 31)
Total raw unique households: 4375
Final analysis sample households: 4309
Total excluded households identified: 66
Cleaned DataFrame shape after removal: (11996256, 31)


In [42]:
# ==========================================
# EXTRACTING EXCLUDED CONTROL HOUSEHOLD IDS
# ==========================================

# 1. Households in the final regression analysis (3,103 treatment + 643 matched controls = 3,746)
# Typically, the paper used all 3,103 treatment households ('OK') but a subset of control households.
# Let's identify control households present in the full control pool versus those retained in df_phase2.

df_treatment_ok = (
    df
    .filter(
        (pl.col("Participation_Phase") == "Phase_2") & 
        (pl.col("Participation_status_Phase2") == "OK")
    )
    .select("ID")
    .unique()
)

df_control_all = (
    df
    .filter(
        (pl.col("Participation_Phase") == "Phase_2") & 
        (pl.col("Control_Price_Phase2") == "Control group")
    )
    .select("ID")
    .unique()
)

# 2. If your df_phase2 or regression subset uses a specific control group subset (643 households),
# you can find the excluded control IDs using an anti-join against your active regression control set:
# (Replace `df_regression_controls` with your actual filtered control dataframe variable if different)

# Example assuming df_phase2 contains 3,746 unique households:
analysis_sample_ids = df_phase2.select("ID").unique()

# Excluded households across the entire dataset (bringing total down from 4,375 to 3,746)
excluded_households = df.select("ID").unique().join(analysis_sample_ids, on="ID", how="anti")

excluded_household_list = excluded_households.get_column("ID").to_list()

print(f"Total unique households in raw dataset: {df.select('ID').n_unique():,}")
print(f"Households in final analysis sample: {analysis_sample_ids.height:,}")
print(f"Total excluded/sub-sampled control households: {len(excluded_household_list):,}")

# 3. Filter your dataframe to remove them
df_cleaned = df.filter(~pl.col("ID").is_in(excluded_household_list))
print(f"Cleaned DataFrame rows: {df_cleaned.height:,}")

Total unique households in raw dataset: 4,375
Households in final analysis sample: 4,309
Total excluded/sub-sampled control households: 66
Cleaned DataFrame rows: 11,996,256


In [23]:
# --- FIGURE B1: Distribution of Hourly Demand & Outdoor Temperature by Region ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
pdf_sample = df_phase2.select(["Region", "Demand_kWh", "Temperature24"]).sample(n=50000, seed=42, with_replacement=False).to_pandas()

sns.boxplot(data=pdf_sample, x="Region", y="Demand_kWh", ax=axes[0], hue="Region", palette="Blues", legend=False)
axes[0].set_title("Hourly Electricity Demand by Region (Phase 2)")
axes[0].set_xlabel("Region")
axes[0].set_ylabel("Demand (kWh)")
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=pdf_sample, x="Region", y="Temperature24", ax=axes[1], hue="Region", palette="coolwarm", legend=False)
axes[1].set_title("24-Hour Rolling Temperature by Region (°C)")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Temperature (°C)")
axes[1].tick_params(axis='x', rotation=30)

fig_path_b1 = os.path.join(parent, "Data", "Figure_B1_Demand_Temperature_Distribution.png")
plt.savefig(fig_path_b1, dpi=300, bbox_inches="tight")
plt.close()
print(f"\nSaved Figure B1 to: {fig_path_b1}")


Saved Figure B1 to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B1_Demand_Temperature_Distribution.png


In [43]:
# --- FIGURE B2: Diurnal Load Profiles (Experiment vs Non-Experiment) ---
df_diurnal = (
    df_phase2
    .with_columns(
        Is_Experiment = pl.col("Experiment_price_NOK_kWh").is_not_null()
    )
    .group_by(["Hour", "Is_Experiment"])
    .agg(
        Mean_Demand = pl.col("Demand_kWh").mean()
    )
    .sort(["Hour", "Is_Experiment"])
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
#palette_dict = {False: "#1b9e77", True: "#d95f02"}

# 1. Define your custom names and colors
label_map = {False: "Non-Experiment Baseline (Control)", True: "Active Price Experiment Day"}
palette_dict = {"Non-Experiment Baseline (Control)": "#1b9e77", "Active Price Experiment Day": "#d95f02"}

# 2. Create a temporary column with the descriptive names
plot_data = df_diurnal.assign(Group=df_diurnal["Is_Experiment"].map(label_map))

# 3. Plot using the new column for hue
sns.lineplot(
    data=plot_data, 
    x="Hour", 
    y="Mean_Demand", 
    hue="Group", 
    palette=palette_dict,
    marker="o", 
    linewidth=2.5,
    ax=ax
)

ax.set_title("Mean Diurnal Load Profile: Experiment Days vs. Non-Experiment Days")
ax.set_xlabel("Hour of Day (1–24)")
ax.set_ylabel("Mean Hourly Electricity Demand (kWh)")
ax.set_xticks(range(1, 25))

# Explicit legend handles with distinct colors to avoid matching line colors directly
# legend_handles = [
#     mlines.Line2D([], [], color="#1b9e77", marker='s', linewidth=2.5, label="Non-Experiment Baseline (Control)"),
#     mlines.Line2D([], [], color="#d95f02", marker='^', linewidth=2.5, label="Active Price Experiment Day")
# ]
# ax.legend(handles=legend_handles, title="Experimental Condition", loc="upper left")

fig_path_b2 = os.path.join(parent, "Data", "Figure_B2_Diurnal_Load_Profiles.png")
plt.savefig(fig_path_b2, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved Figure B2 to: {fig_path_b2}")

Saved Figure B2 to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B2_Diurnal_Load_Profiles.png


In [46]:
type(df_diurnal)

pandas.DataFrame

In [17]:
# ==========================================
# FACETED GRID: DIURNAL PROFILES BY PROFILE & LEVEL
# ==========================================
print("Aggregating diurnal data by Price Profile and Price Level...")

# 1. Prepare data with Price_profile and Price_level indicators
df_faceted_diurnal = (
    df_phase2
    .with_columns(
        Is_Experiment = pl.col("Experiment_price_NOK_kWh").is_not_null()
    )
    # Filter out null profiles/levels if any control rows lack them
    .filter(pl.col("Price_profile").is_not_null() & pl.col("Price_level").is_not_null())
    .group_by(["Hour", "Price_profile", "Price_level", "Is_Experiment"])
    .agg(
        Mean_Demand = pl.col("Demand_kWh").mean()
    )
    .sort(["Price_profile", "Price_level", "Hour", "Is_Experiment"])
    .to_pandas()
)

# 2. Set up FacetGrid
palette_dict = {False: "#1b9e77", True: "#d95f02"}

g = sns.FacetGrid(
    df_faceted_diurnal, 
    col="Price_profile", 
    row="Price_level", 
    hue="Is_Experiment", 
    palette=palette_dict,
    height=3, 
    aspect=1.2, 
    sharey=False  # Allow individual Y-scales since baseline demands vary by price tier
)

# 3. Map line plots onto the grid
g.map_dataframe(sns.lineplot, x="Hour", y="Mean_Demand", marker="o", linewidth=2)

# 4. Formatting and titles
g.set_axis_labels("Hour of Day (1–24)", "Mean Demand (kWh)")
g.add_legend(title="Is Experiment", loc="upper right")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Diurnal Load Profiles Across Price Profiles and Peak Price Levels")

# Save figure
fig_path_grid = os.path.join(parent, "Data", "Figure_B2_Faceted_Diurnal_Profiles.png")
g.savefig(fig_path_grid, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved faceted diurnal grid to: {fig_path_grid}")

Aggregating diurnal data by Price Profile and Price Level...
Saved faceted diurnal grid to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B2_Faceted_Diurnal_Profiles.png


In [22]:
# ==========================================
# FACETED GRID: TREATMENT VS. CONTROL OVERLAY BY PROFILE & LEVEL
# ==========================================
print("Preparing faceted comparison of Treatment vs. Control across Price Profiles and Levels...")

# 1. Extract the mapping of Date, Region, Hour to Price Profile and Price Level from treatment rows
df_signal_mapping = (
    df
    .filter(
        (pl.col("Participation_Phase") == "Phase_2") &
        (pl.col("Experiment_price_NOK_kWh").is_not_null()) &
        (pl.col("Price_profile").is_not_null()) &
        (pl.col("Price_level").is_not_null())
    )
    .select(["Date", "Region", "Hour", "Price_profile", "Price_level"])
    .unique()
)

# 2. Aggregate Treatment group demand on experiment days mapped to their specific signals
df_treatment_diurnal = (
    df
    .filter(
        (pl.col("Participation_Phase") == "Phase_2") &
        (pl.col("Participation_status_Phase2") == "OK") &
        (pl.col("Experiment_price_NOK_kWh").is_not_null())
    )
    .join(df_signal_mapping, on=["Date", "Region", "Hour", "Price_profile", "Price_level"], how="inner")
    .group_by(["Price_profile", "Price_level", "Hour"])
    .agg(Mean_Demand = pl.col("Demand_kWh").mean())
    .with_columns(Condition = pl.lit("Active Price Experiment Day"))
)

# 3. Aggregate Control group demand on those exact same experiment days/regions matched by signal
df_control_diurnal = (
    df
    .filter(
        (pl.col("Participation_Phase") == "Phase_2") &
        (pl.col("Participation_status_Phase2") == "Control")
    )
    .join(df_signal_mapping, on=["Date", "Region", "Hour"], how="inner")
    .group_by(["Price_profile", "Price_level", "Hour"])
    .agg(Mean_Demand = pl.col("Demand_kWh").mean())
    .with_columns(Condition = pl.lit("Non-Experiment Baseline (Control)"))
)

# 4. Combine datasets for plotting and convert to Pandas
df_faceted_overlay = pl.concat([df_treatment_diurnal, df_control_diurnal]).to_pandas()

# 5. Define color palette (Green for Control, Brown for Treatment)
palette_dict = {
    "Non-Experiment Baseline (Control)": "#1b9e77", 
    "Active Price Experiment Day": "#d95f02"
}

# 6. Set up FacetGrid
g = sns.FacetGrid(
    df_faceted_overlay, 
    col="Price_profile", 
    row="Price_level", 
    hue="Condition", 
    palette=palette_dict,
    height=3, 
    aspect=1.2, 
    sharey=False  # Allow individual Y-scales since baseline demand varies across price tiers
)

# 7. Map line plots onto the grid
g.map_dataframe(sns.lineplot, x="Hour", y="Mean_Demand", marker="o", linewidth=2)

# 8. Formatting, labels, and legends
g.set_axis_labels("Hour of Day (1–24)", "Mean Hourly Demand (kWh)")
g.add_legend(title="Experimental Condition", loc="upper right")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Diurnal Load Profiles by Price Profile and Peak Price Level: Treatment vs. Control Overlay")

# Save figure
fig_path_overlay_grid = os.path.join(parent, "Data", "Figure_B3_Faceted_Control_Overlay_Grid.png")
g.savefig(fig_path_overlay_grid, dpi=300, bbox_inches="tight")
plt.close()

print(f"\nSaved faceted control overlay grid to: {fig_path_overlay_grid}")

Preparing faceted comparison of Treatment vs. Control across Price Profiles and Levels...

Saved faceted control overlay grid to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B3_Faceted_Control_Overlay_Grid.png


In [36]:
# ==========================================
# FACETED GRID: TREATMENT VS. CONTROL OVERLAY BY PRICE PROFILE & LEVEL
# ==========================================

# 1. Get unique sorted lists of profiles and levels
profiles = sorted(df_phase2.filter(pl.col("Price_profile").is_not_null())["Price_profile"].unique().to_list())
levels = sorted(df_phase2.filter(pl.col("Price_level").is_not_null())["Price_level"].unique().to_list())

# 2. Set up the matplotlib subplot grid
fig, axes = plt.subplots(len(levels), len(profiles), figsize=(18, 20), sharex=True, sharey=False)

# Define label map and colors matching Figure B2 exactly
label_map = {False: "Non-Experiment Baseline (Control)", True: "Active Price Experiment Day"}
palette_dict = {
    "Non-Experiment Baseline (Control)": "#1b9e77", 
    "Active Price Experiment Day": "#d95f02"
}

# 3. Loop through each price level and profile combination
for r_idx, level in enumerate(levels):
    for c_idx, profile in enumerate(profiles):
        ax = axes[r_idx, c_idx] if len(levels) > 1 else axes[c_idx]
        
        # Filter treatment rows matching this specific profile & level
        df_exp_sub = (
            df_phase2
            .filter(
                (pl.col("Participation_status_Phase2") == "OK") &
                (pl.col("Experiment_price_NOK_kWh").is_not_null()) &
                (pl.col("Price_profile") == profile) &
                (pl.col("Price_level") == level)
            )
        )
        
        # Extract exact Date and Region keys as strings to guarantee join compatibility
        signal_keys = df_exp_sub.select([
            pl.col("Date").cast(pl.Utf8),
            pl.col("Region").cast(pl.Utf8)
        ]).unique()
        
        # Filter Control group matching those exact keys
        df_ctrl_sub = (
            df
            .filter(
                (pl.col("Participation_Phase") == "Phase_2") &
                (pl.col("Participation_status_Phase2") == "Control")
            )
            .with_columns([
                pl.col("Date").cast(pl.Utf8),
                pl.col("Region").cast(pl.Utf8)
            ])
            .join(signal_keys, on=["Date", "Region"], how="inner")
        )
        
        # Aggregate Experiment hourly means
        df_exp_agg = (
            df_exp_sub
            .group_by("Hour")
            .agg(Mean_Demand = pl.col("Demand_kWh").mean())
            .with_columns(Is_Experiment = pl.lit(True))
        )
        
        # Aggregate Control hourly means
        df_ctrl_agg = (
            df_ctrl_sub
            .group_by("Hour")
            .agg(Mean_Demand = pl.col("Demand_kWh").mean())
            .with_columns(Is_Experiment = pl.lit(False))
        )
        
        # Combine into a single pandas dataframe for plotting
        sub_plot_data = (
            pl.concat([df_exp_agg, df_ctrl_agg])
            .sort(["Hour", "Is_Experiment"])
            .to_pandas()
        )
        
        # Map boolean to descriptive legend labels
        sub_plot_data["Group"] = sub_plot_data["Is_Experiment"].map(label_map)
        
        # Plot both lines if data exists
        if not sub_plot_data.empty:
            sns.lineplot(
                data=sub_plot_data,
                x="Hour",
                y="Mean_Demand",
                hue="Group",
                palette=palette_dict,
                marker="o",
                linewidth=2,
                ax=ax,
                legend=False  # Suppress per-subplot legends in favor of a clean global legend
            )
            
        ax.set_title(f"Level: {level} | Profile: {profile}", fontsize=10)
        ax.set_xlabel("Hour (1–24)" if r_idx == len(levels)-1 else "")
        ax.set_ylabel("Demand (kWh)" if c_idx == 0 else "")
        ax.set_xticks(range(1, 25, 5))

# 4. Add global legend handles and figure title matching Figure B2 styling
handles = [
    mlines.Line2D([], [], color="#1b9e77", marker='o', linewidth=2.5, label="Non-Experiment Baseline (Control)"),
    mlines.Line2D([], [], color="#d95f02", marker='o', linewidth=2.5, label="Active Price Experiment Day")
]
fig.legend(handles=handles, title="Experimental Condition", loc="upper right", bbox_to_anchor=(0.98, 0.98))
fig.suptitle("Diurnal Load Profiles Across All Price Profiles & Levels: Treatment vs. Control", fontsize=16, y=0.99)

plt.tight_layout(rect=[0, 0, 1, 0.97])

# Save figure
fig_path_loop_grid = os.path.join(parent, "Data", "Figure_B5_Faceted_Control_Overlay_Grid.png")
plt.savefig(fig_path_loop_grid, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved faceted control overlay grid to: {fig_path_loop_grid}")

Saved faceted control overlay grid to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B5_Faceted_Control_Overlay_Grid.png
